# CloudGuard-Water: Phase 6 — Cloud Prediction Microservice & REST API

**Course**: Machine Learning for Cloud Services  
**Project**: CloudGuard-Water: Intelligent and Explainable Water Quality Classification  
**Notebook**: `06_phase6_cloud_api.ipynb`  
**Author**: Lead ML Deployment Engineer & Python Backend Specialist  
**Date**: October 2026  

---

## 1. Phase 6 Objectives & Deployment Architecture

In Phase 5, we verified the trained **Decision Tree (`max_depth=5`)** as our primary production model, extracted its deterministic decision rules, evaluated SHAP feature attribution on the Random Forest ensemble, and established rigorous non-certifying advisory logic.

The objective of **Phase 6** is to transition from offline model evaluation to an online, production-grade **Cloud REST Microservice**:
1. **Lightweight Concurrency**: Implement an asynchronous REST API using **FastAPI**, **Pydantic v2**, and **Uvicorn**.
2. **Defensive Input Validation**: Enforce strict physical domain constraints (e.g., $0 \le pH \le 14$, non-negative TDS and turbidity), reject corrupt inputs (NaN, infinite floats, missing fields, extra fields), and generate informational warnings for out-of-training-distribution measurements.
3. **Core Endpoints**:
   - `GET /health`: Readiness probe verifying API availability and active in-memory model loading. Returns HTTP 503 if artifacts are missing.
   - `POST /predict`: Real-time potability classification (`Safe` vs. `Unsafe`) with calibrated empirical probabilities and advisory disclaimers.
   - `POST /explain`: Decision path extraction exposing exact node thresholds and human-readable explanations.
4. **Defensive Guardrails**: Ensure live inference never certifies water as potable or safe to drink, explicitly disclaim detection of biological pathogens or heavy metals, and forbid automatic equipment actuation.
5. **In-Process Testing**: Validate endpoints end-to-end using FastAPI's `TestClient` within this notebook.


## 2. Model Artifact Integrity & Schema Verification

We verify that the primary model artifact `models/decision_tree.joblib` exists, loads cleanly, and maintains strict alignment with the canonical feature order `['pH', 'TDS_ppm', 'Turbidity_NTU', 'Temperature_C']` and target classes `[0, 1]` (`0 = Safe`, `1 = Unsafe`).


In [1]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

# Configure project root in path
root_dir = Path("..").resolve() if Path("..").resolve().name == "LB_Proj" else Path(".").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.explain import FEATURE_NAMES, CLASS_LABELS, load_model

model_path = root_dir / "models" / "decision_tree.joblib"
assert model_path.exists(), f"Primary model artifact missing at {model_path}"

dt = load_model(model_path)
print("=" * 80)
print("PRIMARY MODEL ARTIFACT VERIFICATION")
print("=" * 80)
print(f"Model Path           : {model_path}")
print(f"Model Class          : {type(dt).__name__}")
print(f"Max Depth Parameter  : {dt.max_depth} (Actual Depth: {dt.get_depth()})")
print(f"Total Leaves         : {dt.get_n_leaves()} leaves (Total Nodes: {dt.tree_.node_count})")
print(f"Target Classes       : {dt.classes_} -> {[CLASS_LABELS[int(c)] for c in dt.classes_]}")
print(f"Input Feature Names  : {list(dt.feature_names_in_)}")
print("Verification complete: Model schema is 100% compliant with production pipeline.")


PRIMARY MODEL ARTIFACT VERIFICATION
Model Path           : D:\E0224033\ML_for_Cl_Ser\CL\LB_Proj\models\decision_tree.joblib
Model Class          : DecisionTreeClassifier
Max Depth Parameter  : 5 (Actual Depth: 5)
Total Leaves         : 14 leaves (Total Nodes: 27)
Target Classes       : [0 1] -> ['Safe', 'Unsafe']
Input Feature Names  : ['pH', 'TDS_ppm', 'Turbidity_NTU', 'Temperature_C']
Verification complete: Model schema is 100% compliant with production pipeline.


## 3. Microservice Architecture & Pydantic Validation

The API is implemented in [`api/main.py`](file:///d:/E0224033/ML_for_Cl_Ser/CL/LB_Proj/api/main.py). It uses Pydantic to enforce data hygiene before requests reach the scikit-learn model:
- `pH`: Float in range $[0.0, 14.0]$
- `TDS_ppm`: Non-negative float $\ge 0.0$
- `Turbidity_NTU`: Non-negative float $\ge 0.0$
- `Temperature_C`: Float in range $[-10.0, 100.0]^\circ\text{C}$
- `extra="forbid"`: Rejects unknown or unauthorized payload fields.
- `validate_finite`: Explicitly rejects `NaN`, `+Inf`, and `-Inf`.


In [2]:
from api.main import (
    WaterQualityInput,
    PredictionResponse,
    ExplanationResponse,
    HealthResponse,
    TRAINING_DISTRIBUTION_RANGES,
    SAFETY_NOTICE,
)

print("Canonical Training Distribution Ranges:")
for feat, (low, high) in TRAINING_DISTRIBUTION_RANGES.items():
    print(f"  - {feat:15s}: [{low:.2f}, {high:.2f}]")

print("\nStandardized Non-Certification Safety Notice:")
print(f"  '{SAFETY_NOTICE}'")


Canonical Training Distribution Ranges:
  - pH             : [6.09, 9.96]
  - TDS_ppm        : [28.60, 2806.10]
  - Turbidity_NTU  : [0.00, 72.75]
  - Temperature_C  : [16.56, 31.51]

Standardized Non-Certification Safety Notice:
  'This output is a statistical machine learning classification based on 4 physical sensor inputs (pH, TDS, turbidity, temperature), not a certified drinking-water potability declaration. Physical sensors cannot detect microbiological pathogens (e.g., E. coli, Salmonella, noroviruses), toxic heavy metals (e.g., arsenic, lead, mercury), or hazardous organic chemicals. Standardized laboratory testing under statutory drinking-water standards (such as WHO guidelines or BIS IS 10500) remains strictly necessary to establish actual potability. Do not automatically actuate physical equipment (valves, pumps) based solely on this prediction.'


## 4. In-Process TestClient Setup

We initialize FastAPI's `TestClient` in-process. This allows us to dispatch real HTTP requests through the complete Starlette/FastAPI middleware pipeline, testing validation, serialization, and model inference without spawning an external server process.


In [3]:
from fastapi.testclient import TestClient
from api.main import app, load_api_model, app_state

# Ensure model artifact is loaded in application state
load_api_model()
client = TestClient(app)

print(f"FastAPI Application Initialized: '{app.title}' (v{app.version})")
print(f"Active in-memory model: {app_state['model_name']} ({app_state['model_path']})")


FastAPI Application Initialized: 'CloudGuard-Water Prediction & Explainability API' (v1.0.0)
Active in-memory model: DecisionTreeClassifier (models/decision_tree.joblib)


C:\Users\Swaminathan\AppData\Local\Programs\Python\Python310\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 5. Health Check Endpoint (`GET /health`)

Kubernetes and cloud container services (AWS ECS, GCP Cloud Run) require readiness probes to determine whether an instance is capable of serving traffic. We test `GET /health` to confirm active model loading.


In [4]:
resp = client.get("/health")
print(f"HTTP Status Code: {resp.status_code} ({resp.reason_phrase})")
print("Response Payload:")
import json
print(json.dumps(resp.json(), indent=2))
assert resp.status_code == 200
assert resp.json()["status"] == "healthy"
assert resp.json()["model_loaded"] is True


HTTP Status Code: 200 (OK)
Response Payload:
{
  "status": "healthy",
  "model_loaded": true,
  "model_name": "DecisionTreeClassifier",
  "api_version": "1.0.0",
  "classes": [
    "Safe",
    "Unsafe"
  ],
  "features": [
    "pH",
    "TDS_ppm",
    "Turbidity_NTU",
    "Temperature_C"
  ],
  "detail": null
}


## 6. Live Prediction (`POST /predict`): Safe Conforming Water Sample

We submit Row 0 from the canonical dataset, representing potable water complying with physical sensor thresholds:
- `pH`: 7.38
- `TDS_ppm`: 138.7 ppm
- `Turbidity_NTU`: 0.25 NTU
- `Temperature_C`: 20.25°C


In [5]:
safe_payload = {
    "pH": 7.38,
    "TDS_ppm": 138.7,
    "Turbidity_NTU": 0.25,
    "Temperature_C": 20.25,
}

resp_safe = client.post("/predict", json=safe_payload)
print(f"HTTP Status Code: {resp_safe.status_code}")
print("Response JSON:")
print(json.dumps(resp_safe.json(), indent=2))

data = resp_safe.json()
assert data["model_classification"] == "Safe"
assert data["estimated_probabilities"]["Safe"] == 1.0
assert data["estimated_probabilities"]["Unsafe"] == 0.0
assert len(data["warnings"]) == 0


HTTP Status Code: 200
Response JSON:
{
  "model_classification": "Safe",
  "estimated_probabilities": {
    "Safe": 1.0,
    "Unsafe": 0.0
  },
  "model": "DecisionTreeClassifier(max_depth=5)",
  "warnings": [],
  "safety_notice": "This output is a statistical machine learning classification based on 4 physical sensor inputs (pH, TDS, turbidity, temperature), not a certified drinking-water potability declaration. Physical sensors cannot detect microbiological pathogens (e.g., E. coli, Salmonella, noroviruses), toxic heavy metals (e.g., arsenic, lead, mercury), or hazardous organic chemicals. Standardized laboratory testing under statutory drinking-water standards (such as WHO guidelines or BIS IS 10500) remains strictly necessary to establish actual potability. Do not automatically actuate physical equipment (valves, pumps) based solely on this prediction."
}


## 7. Live Prediction (`POST /predict`): Contaminated Water Sample

We submit Row 30240, representing contaminated water exceeding the learned physical thresholds ($TDS = 370.1 > 211.05$ ppm, $\text{Turbidity} = 11.13 > 4.49$ NTU):
- `pH`: 6.89
- `TDS_ppm`: 370.1 ppm
- `Turbidity_NTU`: 11.13 NTU
- `Temperature_C`: 22.25°C


In [6]:
unsafe_payload = {
    "pH": 6.89,
    "TDS_ppm": 370.1,
    "Turbidity_NTU": 11.13,
    "Temperature_C": 22.25,
}

resp_unsafe = client.post("/predict", json=unsafe_payload)
print(f"HTTP Status Code: {resp_unsafe.status_code}")
print("Response JSON:")
print(json.dumps(resp_unsafe.json(), indent=2))

data = resp_unsafe.json()
assert data["model_classification"] == "Unsafe"
assert data["estimated_probabilities"]["Safe"] == 0.0
assert data["estimated_probabilities"]["Unsafe"] == 1.0
assert len(data["warnings"]) == 0


HTTP Status Code: 200
Response JSON:
{
  "model_classification": "Unsafe",
  "estimated_probabilities": {
    "Safe": 0.0,
    "Unsafe": 1.0
  },
  "model": "DecisionTreeClassifier(max_depth=5)",
  "warnings": [],
  "safety_notice": "This output is a statistical machine learning classification based on 4 physical sensor inputs (pH, TDS, turbidity, temperature), not a certified drinking-water potability declaration. Physical sensors cannot detect microbiological pathogens (e.g., E. coli, Salmonella, noroviruses), toxic heavy metals (e.g., arsenic, lead, mercury), or hazardous organic chemicals. Standardized laboratory testing under statutory drinking-water standards (such as WHO guidelines or BIS IS 10500) remains strictly necessary to establish actual potability. Do not automatically actuate physical equipment (valves, pumps) based solely on this prediction."
}


## 8. Live Prediction (`POST /predict`): Borderline Sample (Row 55037)

We submit Row 55037 from the held-out test partition:
- `pH`: 7.05
- `TDS_ppm`: 192.3 ppm
- `Turbidity_NTU`: 3.95 NTU
- `Temperature_C`: 24.26°C

In offline evaluation, this was a known retrospective false negative because its ground-truth label in the dataset generator was `Unsafe`. However, in live production inference:
1. Ground truth is **unknown** at inference time and must never be supplied by the client to alter model outputs.
2. The model outputs its true statistical prediction based on learned training leaf distributions: Leaf #6 has $99.48\%$ Safe and $0.52\%$ Unsafe.
3. The accompanying safety notice ensures that operators never treat a `Safe` prediction as certification of potability.


In [7]:
borderline_payload = {
    "pH": 7.05,
    "TDS_ppm": 192.3,
    "Turbidity_NTU": 3.95,
    "Temperature_C": 24.26,
}

resp_borderline = client.post("/predict", json=borderline_payload)
print(f"HTTP Status Code: {resp_borderline.status_code}")
print("Response JSON:")
print(json.dumps(resp_borderline.json(), indent=2))

data = resp_borderline.json()
assert data["model_classification"] == "Safe"
assert data["estimated_probabilities"]["Safe"] == 0.9948
assert data["estimated_probabilities"]["Unsafe"] == 0.0052


HTTP Status Code: 200
Response JSON:
{
  "model_classification": "Safe",
  "estimated_probabilities": {
    "Safe": 0.9948,
    "Unsafe": 0.0052
  },
  "model": "DecisionTreeClassifier(max_depth=5)",
  "warnings": [],
  "safety_notice": "This output is a statistical machine learning classification based on 4 physical sensor inputs (pH, TDS, turbidity, temperature), not a certified drinking-water potability declaration. Physical sensors cannot detect microbiological pathogens (e.g., E. coli, Salmonella, noroviruses), toxic heavy metals (e.g., arsenic, lead, mercury), or hazardous organic chemicals. Standardized laboratory testing under statutory drinking-water standards (such as WHO guidelines or BIS IS 10500) remains strictly necessary to establish actual potability. Do not automatically actuate physical equipment (valves, pumps) based solely on this prediction."
}


## 9. Explainability Endpoint (`POST /explain`)

The `/explain` endpoint provides complete transparency for water system operators. It reveals the exact Decision Tree node path, learned thresholds, leaf node distribution, and the human-readable advisory narrative generated by `src/explain.py`.


In [8]:
resp_explain = client.post("/explain", json=safe_payload)
print(f"HTTP Status Code: {resp_explain.status_code}")
exp_data = resp_explain.json()

print(f"Model Classification : {exp_data['model_classification']}")
print(f"Terminal Leaf Node   : #{exp_data['leaf_node_id']}")
print(f"Features Utilized    : {exp_data['primary_influencing_features']}")
print("\nDecision Path Traversed:")
for i, s in enumerate(exp_data["decision_path_steps"], 1):
    print(f"  Step {i}: {s['rule']} (Node #{s['node_id']})")

print("\nHuman-Readable Explanation Narrative:")
print("-" * 70)
print(exp_data["human_explanation"])
print("-" * 70)


HTTP Status Code: 200
Model Classification : Safe
Terminal Leaf Node   : #5
Features Utilized    : ['TDS_ppm', 'Turbidity_NTU']

Decision Path Traversed:
  Step 1: TDS_ppm (138.70) <= 202.75 (Node #0)
  Step 2: Turbidity_NTU (0.25) <= 4.49 (Node #1)
  Step 3: Turbidity_NTU (0.25) <= 4.42 (Node #2)
  Step 4: TDS_ppm (138.70) <= 202.35 (Node #3)
  Step 5: Turbidity_NTU (0.25) <= 3.56 (Node #4)

Human-Readable Explanation Narrative:
----------------------------------------------------------------------
Model Classification: SAFE (Statistical Prediction — Not a Potability Certification)

Estimated Model Probabilities (Empirical Training Proportions — Not Guarantees):
  - Estimated P(Safe)  : 100.00%
  - Estimated P(Unsafe): 0.00%

Decision Path Traversal:
  Traversed 5 decision node(s) to Leaf Node #5:
    1. TDS_ppm (138.70) <= 202.75 (Split at node #0)
    2. Turbidity_NTU (0.25) <= 4.49 (Split at node #1)
    3. Turbidity_NTU (0.25) <= 4.42 (Split at node #2)
    4. TDS_ppm (138.70) <= 

## 10. Training-Range Warning Detection

When sensor measurements are physically valid but fall outside the canonical dataset distribution, the API accepts the payload (HTTP 200) but alerts the operator that predictions carry reduced statistical certainty.


In [9]:
out_of_range_payload = {
    "pH": 10.5,             # Training range: [6.09, 9.96]
    "TDS_ppm": 3200.0,        # Training range: [28.6, 2806.1]
    "Turbidity_NTU": 85.0,    # Training range: [0.0, 72.75]
    "Temperature_C": 35.0,    # Training range: [16.56, 31.51]
}

resp_oor = client.post("/predict", json=out_of_range_payload)
print(f"HTTP Status Code: {resp_oor.status_code}")
oor_data = resp_oor.json()
print("Generated Data-Quality Warnings:")
for w in oor_data["warnings"]:
    print(f"  [!] {w}")

assert resp_oor.status_code == 200
assert len(oor_data["warnings"]) == 4


HTTP Status Code: 200
Generated Data-Quality Warnings:
  [!] Feature 'pH' value 10.5 is outside the canonical training range [6.09, 9.96]. Predictions on out-of-distribution measurements may have reduced statistical reliability.
  [!] Feature 'TDS_ppm' value 3200.0 is outside the canonical training range [28.6, 2806.1]. Predictions on out-of-distribution measurements may have reduced statistical reliability.
  [!] Feature 'Turbidity_NTU' value 85.0 is outside the canonical training range [0.0, 72.75]. Predictions on out-of-distribution measurements may have reduced statistical reliability.
  [!] Feature 'Temperature_C' value 35.0 is outside the canonical training range [16.56, 31.51]. Predictions on out-of-distribution measurements may have reduced statistical reliability.


## 11. Defensive Input Validation & Error Handling (HTTP 422 & HTTP 503)

We test invalid payloads to confirm that bad requests are rejected immediately with standard HTTP 422 Unprocessable Entity responses without crashing the server or leaking stack traces.


In [10]:
invalid_cases = [
    ("Negative pH (-1.5)", {"pH": -1.5, "TDS_ppm": 150.0, "Turbidity_NTU": 2.0, "Temperature_C": 22.0}),
    ("pH > 14.0 (15.5)", {"pH": 15.5, "TDS_ppm": 150.0, "Turbidity_NTU": 2.0, "Temperature_C": 22.0}),
    ("Negative TDS (-50.0)", {"pH": 7.0, "TDS_ppm": -50.0, "Turbidity_NTU": 2.0, "Temperature_C": 22.0}),
    ("Non-numeric String", {"pH": "acidic", "TDS_ppm": 150.0, "Turbidity_NTU": 2.0, "Temperature_C": 22.0}),
    ("Missing Required Feature", {"TDS_ppm": 150.0, "Turbidity_NTU": 2.0, "Temperature_C": 22.0}),
    ("Unauthorized Extra Field", {"pH": 7.0, "TDS_ppm": 150.0, "Turbidity_NTU": 2.0, "Temperature_C": 22.0, "extra_key": "injection"}),
]

print("Executing Defensive Input Validation Tests:")
for desc, bad_payload in invalid_cases:
    res = client.post("/predict", json=bad_payload)
    print(f"  - {desc:30s} -> HTTP {res.status_code} (Rejection Confirmed)")
    assert res.status_code == 422

# Verify Missing Model Artifact Handling (HTTP 503)
print("\nTesting Missing Model Artifact Simulation:")
load_api_model(Path("models/nonexistent_dummy.joblib"))
h_res = client.get("/health")
p_res = client.post("/predict", json=safe_payload)
print(f"  - /health  with missing model -> HTTP {h_res.status_code} (status='{h_res.json()['status']}')")
print(f"  - /predict with missing model -> HTTP {p_res.status_code} (detail='{p_res.json()['detail']}')")
assert h_res.status_code == 503
assert p_res.status_code == 503

# Restore model
load_api_model()


Executing Defensive Input Validation Tests:
  - Negative pH (-1.5)             -> HTTP 422 (Rejection Confirmed)
  - pH > 14.0 (15.5)               -> HTTP 422 (Rejection Confirmed)
  - Negative TDS (-50.0)           -> HTTP 422 (Rejection Confirmed)
  - Non-numeric String             -> HTTP 422 (Rejection Confirmed)
  - Missing Required Feature       -> HTTP 422 (Rejection Confirmed)
  - Unauthorized Extra Field       -> HTTP 422 (Rejection Confirmed)

Testing Missing Model Artifact Simulation:
  - /health  with missing model -> HTTP 503 (status='unhealthy')
  - /predict with missing model -> HTTP 503 (detail='Model artifact is not loaded or currently unavailable. Diagnostic detail: Specified model path does not exist: models\nonexistent_dummy.joblib')


## 12. Automated Test Suite Summary

The automated test suite in `tests/test_api.py` comprises 16 dedicated API unit and integration tests. When combined with the 17 Phase 5 regression tests in `tests/test_explain.py`, the project maintains a **33-test regression suite** with a 100% pass rate.


In [11]:
import unittest

suite = unittest.defaultTestLoader.discover(str(root_dir / "tests"), pattern="test_*.py")
runner = unittest.TextTestRunner(verbosity=1)
result = runner.run(suite)

print("\n" + "=" * 80)
print(f"TEST EXECUTION SUMMARY: Ran {result.testsRun} tests in {result.testsRun} test cases.")
print(f"Failures: {len(result.failures)} | Errors: {len(result.errors)}")
print("=" * 80)
assert len(result.failures) == 0
assert len(result.errors) == 0


.

.

.

.

.

...

.

.

.

----------------------------------------------------------------------
Ran 33 tests in 0.343s

OK



TEST EXECUTION SUMMARY: Ran 33 tests in 33 test cases.
Failures: 0 | Errors: 0


## 13. Local Execution & Interactive Swagger Documentation

### Local Run Command
To start the live microservice with hot-reloading:

```bash
uvicorn api.main:app --reload --host 127.0.0.1 --port 8000
```

### Interactive Documentation URLs
Once launched, inspect and interact with the API via:
- **Interactive Swagger UI**: [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)
- **ReDoc Alternative UI**: [http://127.0.0.1:8000/redoc](http://127.0.0.1:8000/redoc)
- **OpenAPI JSON Schema**: [http://127.0.0.1:8000/openapi.json](http://127.0.0.1:8000/openapi.json)

### Sample `curl` Invocations

#### 1. Readiness Probe
```bash
curl -X GET http://127.0.0.1:8000/health
```

#### 2. Prediction Request
```bash
curl -X POST http://127.0.0.1:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"pH": 7.35, "TDS_ppm": 185.0, "Turbidity_NTU": 2.10, "Temperature_C": 22.50}'
```

#### 3. Explainability Request
```bash
curl -X POST http://127.0.0.1:8000/explain \
     -H "Content-Type: application/json" \
     -d '{"pH": 7.35, "TDS_ppm": 185.0, "Turbidity_NTU": 2.10, "Temperature_C": 22.50}'
```


## 14. Key Findings, Limitations & Phase 7 Roadmap

### Key Deliverables Completed:
1. **Production REST API**: Built modular FastAPI microservice in `api/main.py`.
2. **Defensive Validation**: Strict physical bounds, Pydantic type checking, NaN/Inf rejection, and training-distribution range warnings.
3. **Transparent Explainability**: Exposed exact Decision Tree paths and human-readable narratives via `/explain`.
4. **Comprehensive Test Suite**: 16 API tests + 17 Phase 5 tests = 33 automated tests passing in $<0.4$ seconds with 0 Pyright errors.
5. **Containerization**: Standard `Dockerfile` with multi-stage non-root container support.

### Operational Limitations:
1. **Physical Scope**: The API predicts conformity to learned sensor thresholds; it does **not** certify drinking water potability. Microbiological pathogens and heavy metals are unmeasured.
2. **Local Scope**: The server currently runs locally; public cloud deployment (AWS/GCP/Render) with TLS, reverse proxying, and rate limiting is slated for subsequent deployment phases.

### Phase 7 Roadmap (IoT Integration):
- Configure physical ESP32 microcontroller with analog pH, TDS, turbidity, and DS18B20 temperature sensors.
- Implement Wi-Fi HTTP client on ESP32 to dispatch JSON payloads directly to `POST /predict`.
- Display predictions and warnings on an OLED screen or local dashboard.
